In [ ]:
! pip install google-generativeai

In [2]:
import google.generativeai as genai
from typing import Literal
import json
from datetime import datetime

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
class RequestOrchestrator:
    """
    Orchestrator that determines which agent should handle a user's request.
    Routes between Impact Agent and Script Reviewer.
    """

    # Define the two possible agents
    IMPACT_AGENT = "impact agent"
    SCRIPT_REVIEWER = "script reviewer"

    def __init__(self, api_key: str, model_name: str = "gemini-2.5-flash"):
        """
        Initialize the orchestrator with Gemini API.

        Args:
            api_key: Google API key for Gemini
            model_name: Gemini model to use (default: gemini-2.5-flash)
        """
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model_name)
        print(f"✓ Orchestrator initialized with {model_name}")

    def _create_routing_prompt(self, user_prompt: str) -> str:
        """
        Create the system prompt for routing decisions.
        """
        system_prompt = f"""You are a routing system that determines which agent should handle a user's request.

You have TWO agents available:

1. **Impact Agent**: Handles questions about:
   - Impact, outcomes, effects, consequences, results
   - Benefits, changes, influence of actions/policies/programs
   - Analysis of societal, environmental, or business impacts
   - "What if" scenarios and their implications

2. **Script Reviewer**: Handles questions about:
   - Reviewing, analyzing, or critiquing scripts (movie, TV, theater)
   - Code review and programming help
   - Document review and editing
   - Writing feedback and improvements

USER QUERY: {user_prompt}

Based on the user's query, determine which agent should handle this request.

RESPOND WITH ONLY ONE OF THESE TWO OPTIONS (exactly as written):
- "impact agent"
- "script reviewer"

Your response should contain ONLY the agent name, nothing else."""

        return system_prompt

    def route_request(self, user_prompt: str) -> str:
        """
        Determine which agent should handle the user's request.

        Args:
            user_prompt: The user's input query

        Returns:
            Either "impact agent" or "script reviewer"
        """
        # Create the routing prompt
        prompt = self._create_routing_prompt(user_prompt)

        # Get decision from Gemini
        try:
            response = self.model.generate_content(prompt)
            decision = response.text.strip().lower()

            # Validate the decision
            if self.IMPACT_AGENT in decision:
                return self.IMPACT_AGENT
            elif self.SCRIPT_REVIEWER in decision:
                return self.SCRIPT_REVIEWER
            else:
                # Fallback if unexpected response
                print(f"⚠ Unexpected response: {decision}. Using fallback routing.")
                return self._fallback_routing(user_prompt)

        except Exception as e:
            print(f"⚠ Error during routing: {str(e)}. Using fallback routing.")
            return self._fallback_routing(user_prompt)

    def _fallback_routing(self, user_prompt: str) -> str:
        """
        Fallback routing logic if LLM doesn't return expected format.
        Uses keyword-based matching.
        """
        user_lower = user_prompt.lower()

        # Keywords for script reviewer
        script_keywords = ["script", "review", "code", "analyze", "critique",
                          "feedback", "document", "written", "check", "screenplay",
                          "program", "function", "bug", "error", "syntax"]

        # Keywords for impact agent
        impact_keywords = ["impact", "effect", "outcome", "result", "consequence",
                          "benefit", "change", "influence", "affect", "what if",
                          "implications", "happens", "caused by"]

        script_score = sum(1 for keyword in script_keywords if keyword in user_lower)
        impact_score = sum(1 for keyword in impact_keywords if keyword in user_lower)

        if script_score > impact_score:
            return self.SCRIPT_REVIEWER
        else:
            # Default to impact agent if unclear
            return self.IMPACT_AGENT

    def route_with_details(self, user_prompt: str) -> dict:
        """
        Route request and provide detailed information.

        Returns:
            Dictionary with agent decision and metadata
        """
        agent = self.route_request(user_prompt)

        return {
            "agent": agent,
            "user_prompt": user_prompt,
            "timestamp": datetime.now().isoformat(),
            "routing_successful": True
        }

    def batch_route(self, prompts: list) -> list:
        """
        Route multiple prompts at once.

        Args:
            prompts: List of user queries

        Returns:
            List of routing decisions
        """
        results = []
        for prompt in prompts:
            result = self.route_with_details(prompt)
            results.append(result)
        return results

    def generate_response(self, prompt: str) -> str:
        """
        Generate a response from the model for the given full prompt.

        Args:
            prompt: The complete prompt string (system instructions + user content)

        Returns:
            The model's response text, or a short error message on failure
        """
        try:
            response = self.model.generate_content(prompt)
            return response.text.strip() if response.text else "[Empty response]"
        except Exception as e:
            return f"[Generation error] {str(e)}"

    def route_and_respond(self, user_prompt: str, mission: str = "") -> dict:
        """
        Route the user's request to the appropriate agent, build the full prompt,
        generate a response, and return the result.

        Args:
            user_prompt: The user's input query or submission
            mission: Optional studio mission (used only for Impact agent)

        Returns:
            Dict with agent, response, user_prompt, timestamp
        """
        agent = self.route_request(user_prompt)

        if agent == self.IMPACT_AGENT:
            full_prompt = f"{IMPACT_SYSTEM_PROMPT}\n\n"
            if mission and mission.strip():
                full_prompt += f"Current Studio Mission:\n{mission}\n\n"
            full_prompt += f"User Submission:\n{user_prompt}"
        else:
            full_prompt = f"{SCRIPT_REVIEWER_SYSTEM_PROMPT}\n\nUser input:\n{user_prompt}"

        response_text = self.generate_response(full_prompt)

        return {
            "agent": agent,
            "response": response_text,
            "user_prompt": user_prompt,
            "timestamp": datetime.now().isoformat(),
        }

In [4]:
# Agent system prompts (used after routing to build the full prompt for generation)

IMPACT_SYSTEM_PROMPT = """
You are an editorial advisor focused on community engagement and ethical publication. Read the following piece of writing
and identify the communities, audiences, or individuals who may be directly affected by its themes. Then provide specific,
actionable suggestions for how the author can responsibly and meaningfully reach out to or
support those communities through or alongside the text.

Your feedback must:
- Be concrete (e.g., exact additions, placements, wording ideas, or resources to include).
- Explain why each suggestion is appropriate for the content.
- Address ethical considerations such as harm reduction, accessibility, and care for vulnerable readers when relevant.

Avoid generic advice like "be sensitive" or "raise awareness."

When recommending a resource, make sure to provide a way to access it such as a phone number or website.

Do not suggest any changes to the original content like "consider altering the ending" or "elaborate on this."

When users share a script, concept, or idea, your job is to:
1. Analyze the submission's tone, themes, and emotional depth.
2. Determine the topics that are discussed in the submission.
3. Provide clear reasoning for your evaluation.
4. Provide specific, actionable suggestions for how the author can responsibly and meaningfully reach out to or support those communities through or alongside the text.
5. If any information is provided, integrate it naturally into your evaluation.

Respond in this format:

Topics identified:
(List of topics identified in the submission. Each topic should be listed with at least one quote from the submission that exemplifies the listed topic and at least one specific, actionable recommendation for outreach and support actions)
"""

SCRIPT_REVIEWER_SYSTEM_PROMPT = """
You are a script and document reviewer. Your role is to review scripts (screenplay, theater, TV), code, or other written documents and provide structured, actionable feedback.

Your feedback must:
- Identify strengths and what works well.
- Point out specific issues (e.g., clarity, structure, logic, style, or technical errors).
- Give concrete, actionable suggestions for improvement.
- Be concise and organized (use short sections or bullet points when helpful).

Adapt your review style to the type of submission: screenplay/story feedback should address narrative and character; code review should address correctness, style, and maintainability; general documents should address clarity and effectiveness.
"""

In [5]:
def setup_orchestrator():
    """
    Setup function to initialize the orchestrator.
    Replace 'YOUR_API_KEY' with your actual Google API key.
    """
    # Get API key (recommended: use Colab secrets)
    from google.colab import userdata

    try:
        # Try to get API key from Colab secrets
        API_KEY = userdata.get('GOOGLE_API_KEY')
        print("✓ API key loaded from Colab secrets")
    except:
        # Fallback: manual input
        print("⚠ API key not found in secrets. Please enter manually.")
        API_KEY = input("Enter your Google API Key: ")

    # Initialize orchestrator
    orchestrator = RequestOrchestrator(api_key=API_KEY)
    return orchestrator

In [6]:
def test_orchestrator(orchestrator):
    """
    Test the orchestrator with sample queries.
    """
    test_queries = [
        "What is the impact of climate change on coastal cities?",
        "Can you review this Python script for errors?",
        "How does this policy affect small businesses?",
        "Please analyze this screenplay I wrote",
        "What are the consequences of increasing minimum wage?",
        "Check my code for bugs and improvements",
        "What happens if we implement a 4-day work week?",
        "Review my function that calculates fibonacci numbers"
    ]

    print("=" * 70)
    print("ORCHESTRATOR ROUTING TEST")
    print("=" * 70)

    for query in test_queries:
        result = orchestrator.route_with_details(query)
        print(f"\n📝 Query: {query}")
        print(f"➡️  Routed to: {result['agent'].upper()}")
        print("-" * 70)

In [7]:
if __name__ == "__main__":
    # This will run when you execute the notebook
    print("🚀 Starting Request Orchestrator System\n")

    # Setup
    orchestrator = setup_orchestrator()

    # Demo: route and respond (full agent response)
    print("\n" + "=" * 70)
    print("ROUTE AND RESPOND DEMO")
    print("=" * 70)
    demo_impact = "A short story about a family dealing with loss and finding hope in community support."
    result = orchestrator.route_and_respond(demo_impact)
    print(f"\n📝 Query: {result['user_prompt']}...")
    print(f"➡️  Agent: {result['agent'].upper()}")
    print(f"📄 Response:\n{result['response']}")
    print("-" * 70)

    demo_script = "Can you review this Python script for errors? def fib(n): return n if n < 2 else fib(n-1) + fib(n-2)"
    result2 = orchestrator.route_and_respond(demo_script)
    print(f"\n📝 Query: {result2['user_prompt']}...")
    print(f"➡️  Agent: {result2['agent'].upper()}")
    print(f"📄 Response:\n{result2['response']}")
    print("-" * 70)

    demo_impact1 = "What is the impact of climate change on coastal cities?"
    result3 = orchestrator.route_and_respond(demo_impact1)
    print(f"\n📝 Query: {result3['user_prompt']}...")
    print(f"➡️  Agent: {result3['agent'].upper()}")
    print(f"📄 Response:\n{result3['response']}")
    print("-" * 70)
    print(f"")

    print("\n✅ Orchestrator ready for use!")
    print("\nTo get a full response from the routed agent, use:")
    print('  result = orchestrator.route_and_respond("your query here")')
    print('  print(result["agent"], result["response"])')

🚀 Starting Request Orchestrator System

✓ API key loaded from Colab secrets
✓ Orchestrator initialized with gemini-2.5-flash

ROUTE AND RESPOND DEMO

📝 Query: A short story about a family dealing with loss and finding hope in community support....
➡️  Agent: SCRIPT REVIEWER
📄 Response:
Thank you for outlining the type of story you'd like reviewed!

To provide you with structured, actionable feedback as a script and document reviewer, I need the actual content of the short story.

Please paste or attach your short story about "a family dealing with loss and finding hope in community support." Once you provide the text, I'll be able to offer specific feedback on its:

*   **Narrative Arc:** Pacing, plot development, emotional beats.
*   **Character Development:** Depth, motivations, relatability of the family members and community figures.
*   **Theme:** How effectively loss, hope, and community support are explored.
*   **Clarity & Style:** Prose quality, dialogue, imagery, and overall 